In [5]:
import os, glob, time
import pandas as pd
from pyproj import Transformer
import numpy as np
import rasterio

In [6]:
field = "PPAC-B3"
year = "2021"
if field == "PPAC-B3" and year == "2024":
    dates_list = ["061724", "071124", "072324", "080324", "081324", "081924", "090124", "090824"]
    methods = ["whole", "orthorectified"]
elif field == "PPAC-B3" and year == "2021":
    dates_list = ["060421", "062421", "070121", "070921", "071421", "073021", "080421", "081321", "081721", "082521", "083121", "090721", "091521"]
    methods = ["whole", "orthorectified"]
elif field == "Wallpe":
    dates_list = ["080625", "081325", "081525", "082525", "083025", "091025", "091425"]
    methods = ["whole", "aoi", "orthorectified"]
else:
    raise ValueError("Invalid field name. Please choose either 'PPAC-B3' or 'Wallpe'.")

soil_options = ["no_soil", "raw"]
original_crs = "EPSG:4326"  # WGS 84
target_crs = "EPSG:32616"  # WGS 84 - UTM zone 16N
transformer = Transformer.from_crs(original_crs, target_crs, always_xy=True)

In [9]:
print(f"Processing field: {field}, year: {year}, total dates: {len(dates_list)}, methods: {len(methods)}, soil options: {len(soil_options)}")

Processing field: PPAC-B3, year: 2021, total dates: 13, methods: 2, soil options: 2


In [10]:
13 * 2 * 2 * 135

7020

In [7]:
# concatenate all CSVs into a single dataframe
# Add three columns to indicate date, method and soil option
all_dfs = []
for date in dates_list:
    for method in methods:
        for soil in soil_options:
            csv_path = f"../features/{field}/{year}/{date}_{field}_{method}_{soil}.csv"
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                df["DATE"] = date
                df["METHOD"] = method
                df["SOIL"] = "YES" if soil == "raw" else "NO"
                all_dfs.append(df)
            else:
                print(f"Warning: {csv_path} not found.")
                


df = pd.concat(all_dfs, ignore_index=True)
# Reorder columns to have date, method, soil_option at the front
cols = ["DATE", "METHOD", "SOIL"] + [col for col in df.columns if col not in ["DATE", "METHOD", "SOIL"]]
df = df[cols]

In [8]:
# Convert "081525" to "08152025" in the date column
df["DATE"] = df["DATE"].apply(lambda x: x[:4] + "20" + x[4:])
# Reomve total_pixels and vegetation_pixels columns
df = df.drop(columns=["total_pixels", "vegetation_pixels"])

if field == "Wallpe":
    # AOI_ID add 1 to make it 1-based index
    df["aoi_id"] = df["aoi_id"] + 1

# Rename aoi_id to POINT
df = df.rename(columns={"aoi_id": "POINT"})

# Filter only Red, Blue, Green, RedEdge, NIR, NDVI, RDVI, NDRE, OSAVI, PSRI, MCARI2, ExG, and Canopy Cover column with mean, std, max, min
# match exactly the column names that contain these keywords, and keep the date, method, soil, point, latitude and longitude columns
# ignore gndvi even ndvi are in the column names
columns_to_keep = []
for col in df.columns:
    if col.split("_")[0] in ["red", "blue", "green", "rededge", "nir", "ndvi", "rdvi", "ndre", "osavi", "psri", "mcari2", "exg"] or col == "canopy_cover":
        columns_to_keep.append(col)

    
columns_to_keep = ["DATE", "METHOD", "SOIL", "POINT", "latitude", "longitude"] + columns_to_keep
df = df[columns_to_keep]

# Turn the column names to uppercase
df.columns = [col.upper() for col in df.columns]


# Save the combined dataframe to a new CSV
output_csv = f"../combined_vi_stats_filtered_{field}_{year}.xlsx"
df.to_excel(output_csv, index=False)
print(f"Combined Excel file saved to {output_csv}")

Combined Excel file saved to ../combined_vi_stats_filtered_PPAC-B3_2021.xlsx


In [5]:
df

,DATE,METHOD,SOIL,POINT,LATITUDE,LONGITUDE,CANOPY_COVER,NDVI_MEAN,NDVI_STD,NDVI_MIN,...,REDEDGE_MIN,REDEDGE_MAX,NIR_MEAN,NIR_STD,NIR_MIN,NIR_MAX,OSAVI_MEAN,OSAVI_STD,OSAVI_MIN,OSAVI_MAX
0,08062025,whole,NO,1,40.527065,-87.368784,0.232943,0.892875,0.023569,0.713358,...,0.077667,0.203491,0.570005,0.054440,0.248962,0.730408,0.817132,0.030148,0.554973,0.903513
1,08062025,whole,NO,2,40.527940,-87.368292,0.228283,0.884963,0.026352,0.556817,...,0.076263,0.210449,0.530449,0.052443,0.199432,0.670776,0.798047,0.033597,0.407895,0.878098
2,08062025,whole,NO,3,40.529549,-87.367762,0.278589,0.873333,0.030033,0.489007,...,0.070557,0.265106,0.486202,0.056306,0.179016,0.738434,0.772589,0.037211,0.343540,0.877629
3,08062025,whole,NO,4,40.530466,-87.366793,0.257119,0.869597,0.034995,0.414763,...,0.070618,0.302246,0.490444,0.060077,0.134186,0.784088,0.771164,0.042797,0.280459,0.871440
4,08062025,whole,NO,5,40.527002,-87.366412,0.193213,0.888546,0.025513,0.555284,...,0.070404,0.206024,0.560849,0.053750,0.209747,0.732574,0.810806,0.032391,0.429495,0.885557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,09142025,orthorectified,YES,20,40.528159,-87.364109,1.000000,0.845966,0.070904,0.012593,...,0.009247,0.454834,0.369881,0.140477,0.048035,1.060333,0.677459,0.102036,0.010623,0.954106
1004,09142025,orthorectified,YES,21,40.528341,-87.367552,1.000000,0.808672,0.089486,-0.042526,...,0.010986,0.545990,0.366162,0.140426,0.051453,1.050781,0.649696,0.114018,-0.029724,0.949523
1005,09142025,orthorectified,YES,22,40.526676,-87.361435,1.000000,0.831300,0.079142,0.019815,...,0.004974,0.422607,0.306339,0.111831,0.043243,0.813232,0.629847,0.104941,0.014996,0.904161
1006,09142025,orthorectified,YES,23,40.526852,-87.364754,1.000000,0.824775,0.081191,0.029830,...,0.007874,0.364410,0.294037,0.108358,0.045044,0.787781,0.616821,0.106554,0.019202,0.890174


#### Combine disease and VI datasheet

In [6]:
disease_df = pd.read_csv("../Data/PPAC-B3/Disease/2024_PPAC_B3_VisualEvaluations_Mean_Calibrated.csv")
aoi_data = pd.read_csv("../Data/PPAC-B3/PPAC-B3_aoi.csv")
date_info = pd.read_excel("../Data/PPAC-B3/Disease/2024_PPAC_B3_VisualEvaluations_Uncalibrated.xlsx", sheet_name="DAPS")
updated_disease_df = pd.merge(
    disease_df,
    aoi_data[["row", "column", "Point_ID"]],
    left_on=["x", "y"],
    right_on=["row", "column"],
    how="left",
)

In [7]:
date_info

,Time,DAP,Date,UAV Data,Unnamed: 4
0,1,29,June 18 2024,June 17,NaN
1,2,38,June 27 2024,NaN,NaN
2,3,43,July 2 2024,NaN,NaN
3,4,49,July 8 2024,July 11,NaN
4,5,58,July 17 2024,July 17 (no panel),NaN
5,6,64,July 23 2024,July 23,NaN
6,7,75,August 3 2024,August 3,only center plant evaluated
7,8,79,August 7 2024,NaN,NaN
8,9,86,August 14 2024,August 13,NaN
9,10,92,August 20 2024,August 19,NaN


In [8]:
updated_disease_df

,grid,leaf,x,y,tar1,tar2,tar3,tar4,tar5,tar6,...,tar11,tar12,tar13,cn12,cn13,tarcn12,tarcn13,row,column,Point_ID
0,A1,L5,9,1,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,1,121
1,A1,E2,9,1,NaN,NaN,NaN,NaN,0.0,0.0,...,0.716920,6.629137,12.833333,1.926294,29.833333,8.555431,42.666667,9,1,121
2,A1,E2+3,9,1,NaN,NaN,NaN,NaN,0.0,0.0,...,0.392549,3.654752,6.333333,0.000000,5.333333,3.654752,11.666667,9,1,121
3,A2,L5,9,2,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,2,122
4,A2,E2,9,2,NaN,NaN,NaN,NaN,0.0,0.0,...,0.557155,6.758458,12.500000,2.645591,28.666667,9.404050,41.166667,9,2,122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,I14,E2,1,14,NaN,NaN,NaN,NaN,0.0,0.0,...,0.532426,4.799515,12.494682,0.000000,26.807301,4.799515,39.301983,1,14,14
401,I14,E2+3,1,14,NaN,NaN,NaN,NaN,0.0,0.0,...,0.262931,2.740773,6.960098,0.166667,9.723265,2.907439,16.683363,1,14,14
402,I15,L5,1,15,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,15,15
403,I15,E2,1,15,NaN,NaN,NaN,NaN,0.0,0.0,...,0.545466,6.214901,12.112987,0.000000,7.371115,6.214901,19.484102,1,15,15


In [78]:
# Add a disease severity column to df based on POINT and closest date
# Map each UAV flight date to the closest disease evaluation Time index
date_info_clean = date_info.dropna(subset=["UAV Data"]).copy()
date_info_clean["UAV_DATE"] = pd.to_datetime(date_info_clean["UAV Data"] + " 2024", format="%B %d %Y")

df_dates = pd.to_datetime(df["DATE"], format="%m%d%Y")
unique_dates = df_dates.unique()
date_to_time = {
    d: date_info_clean.loc[(date_info_clean["UAV_DATE"] - d).abs().idxmin(), "Time"]
    for d in unique_dates
}
df["_TIME"] = df_dates.map(date_to_time)



In [79]:
# Average severity (tar{Time}) across leaf types for each Point_ID
tar_cols = [c for c in updated_disease_df.columns if c.startswith("tar") and c[3:].isdigit()]
severity_long = updated_disease_df.groupby("Point_ID")[tar_cols].mean().reset_index()
severity_long = severity_long.melt(id_vars="Point_ID", var_name="TAR", value_name="SEVERITY")
severity_long["_TIME"] = severity_long["TAR"].str.replace("tar", "").astype(int)
severity_long = severity_long.drop(columns="TAR")

df = df.merge(severity_long, left_on=["POINT", "_TIME"], right_on=["Point_ID", "_TIME"], how="left")
df = df.drop(columns=["Point_ID", "_TIME"])

In [80]:
df

,DATE,METHOD,SOIL,POINT,LATITUDE,LONGITUDE,CANOPY_COVER,NDVI_MEAN,NDVI_STD,NDVI_MIN,...,REDEDGE_MAX,NIR_MEAN,NIR_STD,NIR_MIN,NIR_MAX,OSAVI_MEAN,OSAVI_STD,OSAVI_MIN,OSAVI_MAX,SEVERITY
0,06172024,whole,NO,1,41.453298,-86.942154,0.346282,0.471544,0.140135,0.236954,...,0.223389,0.213516,0.042826,0.111237,0.417511,0.351677,0.112493,0.196661,0.711184,0.000000
1,06172024,whole,NO,2,41.453296,-86.942013,0.400573,0.478582,0.148998,0.221666,...,0.247894,0.222662,0.045547,0.111176,0.442810,0.361328,0.119840,0.191168,0.718405,0.000000
2,06172024,whole,NO,3,41.453294,-86.941866,0.438713,0.490750,0.147178,0.229131,...,0.228912,0.220941,0.044001,0.111053,0.419983,0.368642,0.117909,0.193705,0.701586,0.000000
3,06172024,whole,NO,4,41.453292,-86.941710,0.454571,0.490470,0.149423,0.220549,...,0.245880,0.218941,0.043658,0.110504,0.407990,0.367233,0.119093,0.191095,0.714900,0.000000
4,06172024,whole,NO,5,41.453290,-86.941568,0.460471,0.503475,0.151430,0.231035,...,0.219727,0.220125,0.045871,0.110168,0.417908,0.376709,0.121886,0.194577,0.716384,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4315,09082024,orthorectified,YES,131,41.453938,-86.940729,1.000000,0.821281,0.062615,0.318010,...,0.314423,0.306458,0.099737,0.044342,0.702820,0.627463,0.098710,0.214169,0.846994,4.133240
4316,09082024,orthorectified,YES,132,41.453936,-86.940584,1.000000,0.822499,0.061764,0.256774,...,0.294220,0.310273,0.100388,0.041870,0.663818,0.631005,0.098568,0.195118,0.853982,3.046943
4317,09082024,orthorectified,YES,133,41.453933,-86.940436,1.000000,0.813570,0.066931,0.230872,...,0.291168,0.301014,0.098979,0.041870,0.662750,0.618627,0.101338,0.168903,0.832533,1.734334
4318,09082024,orthorectified,YES,134,41.453931,-86.940296,1.000000,0.821083,0.060732,0.327069,...,0.301178,0.308786,0.099125,0.045380,0.691101,0.629329,0.097123,0.224176,0.842568,1.650275


In [81]:
df.to_excel(f"../combined_vi_stats_with_disease_{field}.xlsx", index=False)